In [1]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import *
import torch.nn as nn
import sqlite3
import torch.nn.functional as f


In [2]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
# tokenizer = AutoTokenizer.from_pretrained('t5-large')

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-multilingual-cased")

pre_trained_model = AutoModelForMaskedLM.from_pretrained("distilbert/distilbert-base-multilingual-cased")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
tokenizer.vocab_size

119547

In [ ]:
len(tokenizer.all_special_tokens)

5

In [ ]:
english_sentences = ["Hello, how are you?", "This is an English sentence."]
german_sentences = ["Hallo, wie geht es Ihnen?", "Dies ist ein deutscher Satz."]

In [ ]:
english_tokens = [tokenizer.tokenize(token) for token in english_sentences]
german_tokens = [tokenizer.tokenize(token) for token in german_sentences]
english_tokens, german_tokens

In [ ]:
print(list(tokenizer.get_vocab().items()) [21:26])

[('mouvoir', 30089), ('▁results', 772), ('▁Vorlage', 29399), ('fenced', 31037), ('<extra_id_70>', 32029)]


In [ ]:
tokenizer(['français bonjour', 'morning francais']).input_ids

[[125, 3, 9, 786, 239, 55, 1], [1379, 3, 6296, 658, 159, 1]]

In [3]:
de_en_dataset = pd.read_csv("/content/GERMAN_ENGLISH_TRANSLATION.csv")

largest_len_sentence = 0
for sentence in de_en_dataset['GERMAN']:
    largest_len_sentence = max(largest_len_sentence, len(sentence.split(' ')))

In [ ]:
largest_len_sentence

53

In [4]:
class TranslationDataset(Dataset):
  def __init__(self, dataframe, tokenizer):
    self.source_sentences = list(dataframe['GERMAN'])
    self.target_sentences = list(dataframe['ENGLISH'])
    self.tokenizer = tokenizer

  def __len__(self):
    return len(self.source_sentences)

  def __getitem__(self, idx):
    source = self.tokenizer.encode(self.source_sentences[idx], add_special_tokens=False)
    target = self.tokenizer.encode(self.target_sentences[idx], add_special_tokens=True)

    source_tensor = torch.tensor(source)
    target_input_tensor = torch.tensor(target[:-1])
    target_label_tensor = torch.tensor(target[1:])

    return source_tensor, len(source), target_input_tensor, target_label_tensor

  @staticmethod
  def collate_function(batch):
    source, source_length, target_input, target_label = zip(*batch)

    pad_source = torch.nn.utils.rnn.pad_sequence(source,batch_first=True, padding_value=tokenizer.pad_token_id)

    pad_target_input = torch.nn.utils.rnn.pad_sequence(target_input, batch_first=True, padding_value=tokenizer.pad_token_id)
    pad_target_label = torch.nn.utils.rnn.pad_sequence(target_label, batch_first=True, padding_value=tokenizer.pad_token_id)

    source_length = torch.as_tensor(source_length, dtype=torch.long)

    return pad_source, source_length, pad_target_input, pad_target_label





In [ ]:
%pip install lightning

In [6]:
import lightning as L


class TranslationDataModule(L.LightningDataModule):

  def __init__(self, batch_size, num_workers, train_ds):
    super().__init__()
    self.batch_size = batch_size
    self.num_workers = num_workers
    self.train_ds = train_ds


  def setup(self, stage: str):
    self.train_ds, self.val_ds = random_split(
        self.train_ds, (0.8, 0.2))

  def train_dataloader(self):
    return DataLoader(
        self.train_ds,
        batch_size = self.batch_size,
        shuffle = True,
        num_workers=self.num_workers,  # Number of subprocesses
        prefetch_factor=2,             # 2 batches ahead of time
        collate_fn = TranslationDataset.collate_function
    )

  def val_dataloader(self):
    return DataLoader(
        self.val_ds,
        batch_size = self.batch_size,
        num_workers = self.num_workers,
        prefetch_factor=2,
        collate_fn = TranslationDataset.collate_function
    )

    # the parameters prefetch_factor and num_workers are used to control how data loading is performed,
    # particularly in terms of parallelism and efficiency
    # Two subprocesses will be used to load the data simultaneously
    # prefetch_factor=2: Each worker will load 2 batches ahead of time, ensuring that the next batch is available without waiting for the current one to finish.

In [7]:
translation_dataset = TranslationDataset(dataframe=de_en_dataset, tokenizer=tokenizer)

translation_datamodule = TranslationDataModule(
    batch_size=64,
    num_workers=2,
    train_ds=translation_dataset)

translation_datamodule.setup('fit')
translation_datamodule.train_dataloader()

for bch in translation_datamodule.train_dataloader():

  print(bch[1][1])
  print(bch[2][0])
  break



tensor(12)
tensor([  101, 19132, 19556, 10106, 10226, 13000, 10111, 13446, 11816,     0,
            0,     0,     0,     0,     0,     0])


In [19]:
tokenizer.vocab_size

119547

In [8]:
# no pretrained embedding layer & position of embedding layer
# 1. Embedding Layer
vocab_size = 119547  # Example vocabulary size
embedding_dim = 768  # Embedding dimension

embedding_layer = nn.Embedding(vocab_size, embedding_dim)

# 2. position of Embedding Layer

max_seq_len = 512
embedding_dim = 768

pos_embedding_layer = nn.Embedding(max_seq_len, embedding_dim)
pos_embedding_layer = pos_embedding_layer.requires_grad_(True)

print(pos_embedding_layer)

Embedding(512, 768)


In [10]:
# pretrained embedding layer & position of embedding layer
# 1. Embedding Layer
embedding_layer = pre_trained_model.get_input_embeddings()

# These line just tells pytorch we don't intend to further train the embedding layer.
# So it freezes the layers knowledge, so we don't scatter it while our model is still starting to learn.
embedding_layer = embedding_layer.requires_grad_(False)

# 2. position of Embedding Layer

pos_embedding_layer = pre_trained_model.get_position_embeddings()
pos_embedding_layer = pos_embedding_layer.requires_grad_(False) # Freezes knowledge as we have seen before.

pos_embedding_layer # Note how it supports up to 512 tokens and has an embed dim of 768 just like our word embedding layer.

Embedding(512, 768)

In [10]:
class PositionalEncoding(nn.Module):
  def __init__(self, *, pos_embedding_layer):
    super().__init__()
    self.pos_embedding_layer = pos_embedding_layer

  def forward(self, X):
    position_indices = torch.arange(X.size(1), device= X.device)
    positional_embeddings = self.pos_embedding_layer(position_indices)
    return X + positional_embeddings


class MultiHeadAttention(nn.Module):

    def __init__(self, *, dim_qkv, dim_model, num_heads, causal=False, **kwargs):
        super().__init__(**kwargs)

        # Ensure dim_qkv (embedding dimension) is divisible by number of heads
        assert dim_qkv % num_heads == 0, "DIM_QKV must be divisible by num_heads."

        self.num_heads = num_heads
        self.causal = causal

        # Scaling factor for attention scores
        self.scale_factor = torch.math.sqrt(dim_qkv//num_heads)

        # The neural networks for queries, keys, and values from the word embeddings
        self.obtain_queries = nn.Linear(dim_model, dim_qkv, bias=False)
        self.obtain_keys = nn.Linear(dim_model, dim_qkv, bias=False)
        self.obtain_values = nn.Linear(dim_model, dim_qkv, bias=False)

        # Final linear projection to get our updated context information back to the same shape as
        # the input (concise form for the model).
        self.projection = nn.Linear(dim_qkv, dim_model)

    def forward(self, queries, keys, values, seq_lengths=None):

        # Get our queries, keys, and values from the embeddings
        queries = self.obtain_queries(queries)
        keys = self.obtain_keys(keys)
        values = self.obtain_values(values)

        # Reshape for multi-head attention (will shape our matrix as if we create multiple heads separately).
        queries = self.parallel_reshape(queries)
        keys = self.parallel_reshape(keys)
        values = self.parallel_reshape(values)

        # Calculate attention scores (Remember our attention weight matrix 🙂)
        attention = queries @ keys.transpose(-1, -2)

        # Scale scores to prevent saturation (so high scores don't totally drown lower ones)
        attention = attention / self.scale_factor

        # Apply causal mask (this will be down when we pass causal=true in masked multi-head attention)
        if self.causal:
            attention = attention + self.get_causal_mask(queries)
        else:
        # Apply padding mask (this will be perform for only reqular multi-head attention)
        # Note: we don't need padding mask for causal attention because each token can only see previous token
        # so the padded token can't spoil the embeddings of acutual tokens.
            attention = torch.masked_fill(attention, self.get_timeseq_mask(seq_lengths), -torch.inf)

        # Calculate attention distribution (softmax)
        attention = f.softmax(attention, dim=-1)

        # Apply attention distribution to values (Weighted combination of the values to form better contextual embeddings)
        X = attention @ values

        # Reshape back to original (This is similar to us concatenating the information from multiple heads).
        X = self.reverse_reshape(X)

        # Final linear projection (Project back our rich embeddings to same shape as the inputs, a concise form for the model)
        X = self.projection(X)

        return X

    def parallel_reshape(self, tensor):

        Batch_size, Seq_len = tensor.shape[0], tensor.shape[1]
        return tensor.reshape(Batch_size, Seq_len, self.num_heads, -1).permute(0, 2, 1, 3)

    def reverse_reshape(self, tensor):

        Batch_size, Seq_len = tensor.shape[0], tensor.shape[2]
        return tensor.permute(0, 2, 1, 3).reshape(Batch_size, Seq_len, -1)

    def get_timeseq_mask(self, x_lengths):

        max_seq_len = x_lengths.max().item()  # Get the maximum sequence length in the batch

        # Create a sequence of numbers from 0 to max_seq_len (representing positions in the sequence)
        ids = torch.arange(0, max_seq_len, device=x_lengths.device)

        # Broadcast the sequence lengths to create a comparison matrix (batch_size, seq_len)
        # True where a position's id is less than the corresponding sequence length (valid position)
        mask = ids[None, :] < x_lengths[:, None]

        # Invert the mask to obtain True for positions that should be masked (padded tokens)
        return ~mask[:, None, None]


    def get_causal_mask(self, X):

        max_seq_len = X.size(2)  # Get sequence length from the input tensor
        mask = nn.Transformer.generate_square_subsequent_mask(max_seq_len, device=X.device) # generate mask above the main diagonal (like our black tape).
        return mask

In [11]:
class EncoderBlock(nn.Module):
    """
    Represents a single encoder block in a Transformer model with ReZero modification.
    """

    def __init__(self, dim_qkv, dim_model, num_heads, dim_ffn, dropout_rate, **kwargs):
        super().__init__(**kwargs)

        # Multi-Head Self-Attention Layer
        self.multi_head_attn = MultiHeadAttention(
            dim_qkv=dim_qkv,  # Dimension of query, key, and value vectors
            dim_model=dim_model,  # Size of dim_model
            num_heads=num_heads  # Number of attention heads
        )

        # Feed-Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(dim_model, dim_ffn),  # First linear layer
            nn.ReLU(),  # ReLU activation for non-linearity (to aid learning complex patterns)
            nn.Linear(dim_ffn, dim_model)  # Second linear layer
        )

        # Dropout for regularization (forcing the model to utilize as much information as it can get.)
        self.dropout = nn.Dropout(dropout_rate)

        # ReZero parameter (learnable weight for weighted addition, typically intialized to zero)
        self.reZero = nn.Parameter(torch.tensor(0.0))

    def forward(self, X, X_len):
        

        # Skip connection for residual addition
        skip = X

        # Multi-Head Self-Attention with ReZero
        X = self.multi_head_attn(queries=X, keys=X, values=X, seq_lengths=X_len)
        X = self.dropout(X)
        X = skip + self.reZero * X  # ReZero weighted addition

        # Feed-Forward Network with ReZero
        skip = X
        X = self.ffn(X)
        X = self.dropout(X)
        X = skip + self.reZero * X  # ReZero weighted addition

        return X

In [12]:
class EncoderLayer(nn.Module):


    def __init__(self,
                 num_encoder_blocks,
                 dim_qkv,
                 dim_model,
                 num_heads,
                 dim_ffn,
                 dropout_rate,
                 **kwargs) -> None:

        super().__init__(**kwargs)

        # Collection of Encoder Blocks
        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(
                dim_qkv,
                dim_model,
                num_heads,
                dim_ffn,
                dropout_rate
            ) for _ in range(num_encoder_blocks)
        ])

    def forward(self, X, X_len):
        """
        Passes the input through each Encoder Block sequentially.
        """
        for encoder in self.encoder_blocks:
            X = encoder(X, X_len)

        return X

In [13]:
class DecoderBlock(nn.Module):


    def __init__(self, dim_qkv, dim_model, num_heads, dim_ffn, dropout_rate, **kwargs):
        super().__init__(**kwargs)

        # Masked Multi-Head Attention for self-attention within decoder input
        self.masked_multi_head_attn = MultiHeadAttention(
            dim_qkv=dim_qkv,
            dim_model=dim_model,
            num_heads=num_heads,
            causal=True  # Use causal masking to prevent future token peeking
        )

        # Cross Attention to attend to the encoded representation
        self.cross_attn = MultiHeadAttention(
            dim_qkv=dim_qkv,
            dim_model=dim_model,
            num_heads=num_heads
        )

        # Feed-Forward Network just like we saw in encoder block.
        self.ffn = nn.Sequential(
            nn.Linear(dim_model, dim_ffn),
            nn.ReLU(),
            nn.Linear(dim_ffn, dim_model)
        )

        # Dropout for regularization (just like in encoder block).
        self.dropout = nn.Dropout(dropout_rate)

        # ReZero parameter for weighted addition
        self.reZero = nn.Parameter(torch.tensor(0.0))

    def forward(self, X, enc_outputs, enc_seq_lengths):

        # Skip connection for residual addition
        skip = X

        # Masked Multi-Head Attention (Self-Attention)
        # - Prevents peeking at future words during training
        # - Focuses on context within the decoder's input
        X = self.masked_multi_head_attn(queries=X, keys=X, values=X)
        X = self.dropout(X)
        X = skip + self.reZero * X  # ReZero weighted addition

        # Skip connection for residual addition
        skip = X

        # Cross Attention
        # - Attends to the encoded representation for context
        # - Combines information from decoder and encoder
        # - Also pass the encoder sequence lengths so we don't try to get encoded information from [PAD] tokens.
        X = self.cross_attn(queries=X, keys=enc_outputs, values=enc_outputs, seq_lengths=enc_seq_lengths)
        X = self.dropout(X)
        X = skip + self.reZero * X  # ReZero weighted addition

        # Skip connection for residual addition
        skip = X

        # Feed-Forward Network (just like we have seen before).
        X = self.ffn(X)
        X = self.dropout(X)
        X = skip + self.reZero * X  # ReZero weighted addition

        return X

In [14]:
class DecoderLayer(nn.Module):
    

    def __init__(self, num_decoder_blocks, dim_qkv, dim_model, num_heads, dim_ffn, dropout_rate, **kwargs):
        super().__init__(**kwargs)

        # Collection of Decoder Blocks
        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(
                dim_qkv,
                dim_model,
                num_heads,
                dim_ffn,
                dropout_rate
            ) for _ in range(num_decoder_blocks)
        ])

    def forward(self, X, enc_out, enc_seq_lengths):
        """
        Passes the input through each Decoder Block sequentially.
        """
        for decoder in self.decoder_blocks:
            X = decoder(X, enc_out, enc_seq_lengths)

        return X

In [15]:
# Model Hyper-Parameters
DIM_MODEL = 768

DIM_FFN = 784
DIM_QKV = 512

DROPOUT_RATE = 0.2
VOCAB_SIZE = tokenizer.vocab_size

TOKEN_LIMIT = 350

In [16]:
from torchmetrics import Accuracy

class Transformer(L.LightningModule):
  def __init__(self, **kwargs):
    super().__init__()

    # Positional Encoding
    self.pos_embed_layer = PositionalEncoding(
        pos_embedding_layer = pos_embedding_layer
    )

    self.embedding_layer = embedding_layer

    # Inject our pre-trained token embedding layer.
    self.encoder_layer = EncoderLayer(
        num_encoder_blocks=4,
        dim_qkv =DIM_QKV,
        dim_model=DIM_MODEL,
        num_heads=8,
        dim_ffn=DIM_FFN,
        dropout_rate=DROPOUT_RATE
    )

    self.decoder_layer = DecoderLayer(
        num_decoder_blocks=4,
        dim_qkv=DIM_QKV,
        dim_model=DIM_MODEL,
        num_heads=8,
        dim_ffn=DIM_FFN,
        dropout_rate=DROPOUT_RATE
    )

    # output layer
    self.output = nn.Linear(DIM_MODEL, VOCAB_SIZE)
    self.loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

    self.accuracy = Accuracy(
        task = 'multiclass',
        num_classes = VOCAB_SIZE,
        ignore_index = tokenizer.pad_token_id
    )

  def forward(self, enc, enc_len, dec):
    enc = self.embedding_layer(enc)
    dec = self.embedding_layer(dec)

    enc = self.pos_embed_layer(enc)
    dec = self.pos_embed_layer(dec)

    enc = self.encoder_layer(enc, enc_len)

    dec = self.decoder_layer(dec, enc_out=enc, enc_seq_lengths=enc_len)

    logits = self.output(dec)

    return logits

  def training_step(self, batch, batch_idx):
    return self._common_step(batch, 'train')

  def validation_step(self, batch, batch_idx):
    return self._common_step(batch, 'val')

  def _common_step(self, batch, prefix):
    enc, enc_len, x_dec, y_dec = batch

    logits = self(enc, enc_len, x_dec)

    loss = self.loss_fn(logits.permute(0, 2, 1), y_dec)

    self.log_dict({
        f'{prefix} acc': self.accuracy(logits.permute(0, 2, 1), y_dec),
        f'{prefix} loss': loss
    },
      prog_bar=True)

    return loss

  def configure_optimizers(self):
    optimizer = torch.optim.Adam(self.parameters(), lr=5e-4)
    return optimizer


In [17]:
!pip install torchinfo

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from torchinfo import summary

batch_size = 64 # will emulate passing a batch size of 24 through the model.
summary(
    Transformer(),
    input_data = [
        torch.randint(low=2, high=250, size=(batch_size, TOKEN_LIMIT)), # emulate encoder inputs
        torch.full([batch_size], TOKEN_LIMIT), # emulate lengths of encoder inputs
        torch.randint(low=2, high=250, size=(batch_size, TOKEN_LIMIT)) # emulate decoder inputs
    ],
    device=device
)

Layer (type:depth-idx)                        Output Shape              Param #
Transformer                                   [64, 350, 119547]         --
├─Embedding: 1-1                              [64, 350, 768]            91,812,096
├─Embedding: 1-2                              [64, 350, 768]            (recursive)
├─PositionalEncoding: 1-3                     [64, 350, 768]            --
│    └─Embedding: 2-1                         [350, 768]                393,216
├─PositionalEncoding: 1-4                     [64, 350, 768]            (recursive)
│    └─Embedding: 2-2                         [350, 768]                (recursive)
├─EncoderLayer: 1-5                           [64, 350, 768]            --
│    └─ModuleList: 2-3                        --                        --
│    │    └─EncoderBlock: 3-1                 [64, 350, 768]            2,779,409
│    │    └─EncoderBlock: 3-2                 [64, 350, 768]            2,779,409
│    │    └─EncoderBlock: 3-3            

In [19]:
from lightning.pytorch.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='./checkpoints/',
    filename= 'de_en_model',
    save_top_k = 1
)

In [20]:
trainer = L.Trainer(
    max_epochs = 5,
    callbacks = [checkpoint_callback]
)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [21]:
transformer_model = Transformer()

In [22]:
trainer.fit(
    transformer_model, # our transformer instance
    translation_datamodule # the data module we previously created, will be used for training the model.
)

/usr/local/lib/python3.10/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name            | Type               | Params | Mode 
---------------------------------------------------------------
0 | pos_embed_layer | PositionalEncoding | 393 K  | train
1 | embedding_layer | Embedding          | 91.8 M | train
2 | encoder_layer   | EncoderLayer       | 11.1 M | train
3 | decoder_layer   | DecoderLayer       | 17.4 M | train
4 | output          | Linear             | 91.9 M | train
5 | loss_fn         | CrossEntropyLoss   | 0      | train
6 | accuracy        | MulticlassAccuracy | 0      | train
---------------------------------------------------------------
212 M     Trainable params
0         Non-trainable params
212 M     Total params
850.667   Total estimated mod

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


In [23]:
def clean_text(text):

    # Convert text to lowercase
    text = str(text).lower().strip()

    # Remove the \n at the end of each line in the file
    text = text.rstrip('\n')

    # Remove HTML tags and non-alphanumeric characters
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"[^a-zA-ZÀ-ÿ0-9\s.,;!?':()\[\]{}-]", " ", text)  # Keep selected punctuation marks, symbols and apostrophes

    # Remove excessive whitespace (more than one space)
    text = re.sub(r"\s+", " ", text)

    text = text.encode("utf-8", errors="ignore").decode("utf-8")  # Corrected encoding

    return text

In [24]:
check_point_path = "/content/checkpoints/de_en_model.ckpt"
loaded_model = Transformer.load_from_checkpoint(check_point_path)


In [25]:
import regex as re
def naive_greedy_decoding(de_sentence, transformer_model, tokenizer, max_output_len=TOKEN_LIMIT):


    # **Evaluation mode and GPU usage:**
    transformer_model = transformer_model.eval().cuda()  # Switch to evaluation mode and move to GPU

    # **Input validation:**
    assert isinstance(de_sentence, str), "The german sentence should be a string"
    de_sentence = de_sentence.strip()

    if de_sentence == "":
        raise Exception('Text should not be empty')

    de_sentence = clean_text(de_sentence)  # Apply any necessary text cleaning

    # **Tokenization and sequence lengths:**
    de_sen = tokenizer.encode(de_sentence, return_tensors='pt')  # Tokenize English sentence
    de_sen_len = torch.tensor([de_sen.shape[1]])  # Calculate input sequence length

    # **Initialize decoded sentence and loop:**
    en_decoded = [tokenizer.cls_token_id]  # Start with [CLS] token representing our Start Of Sentence
    for _ in range(max_output_len):

        # **Disable gradient calculation for efficiency:**
        with torch.no_grad():
            # **Forward pass through the Transformer:**
            log_proba = transformer_model(
                de_sen.cuda(),  # Input English sentence on GPU
                de_sen_len.cuda(),  # Input English sentence length on GPU
                torch.tensor([en_decoded]).cuda()  # Current decoded sentence on GPU
            )  # Shape: [1, len(fr_decoded), vocab_size]

            # **Greedy decoding: choose word with highest probability**
            next_word_id = torch.argmax(log_proba, dim=-1)[0][-1]
            en_decoded.append(next_word_id)  # Add predicted word to decoded sentence

            # **Early stopping if [SEP] (our end-of-sentence) token is predicted**
            if next_word_id == tokenizer.sep_token_id:
                break

    # **Decode tokens back to human-readable text:**
    en_text = tokenizer.decode(
        en_decoded,
        clean_up_tokenization_spaces=True  # Remove extra spaces around punctuation
    ).replace(' ##', '')  # Join subwords back into complete words

    return en_text

In [31]:
text = """Die Sonne schien hell am Himmel, während die Kinder fröhlich im Park spielten und ihre Drachen steigen ließen,
 die bunt in der Luft tanzten."""

naive_greedy_decoding(
    text,
    loaded_model,
    tokenizer
    )

'[CLS] the sun prevented the earth of the earths of the earth and went to the restaurant and the earth [SEP]'